# sim_cut — Phase 2 extraction (Colab, multi-segment)

Extracts per-segment features for a whole session. The recording camera cuts every ~30 min, so a session is several files; we extract each **separately** (so motion is never differenced across a seam -> no false walk-over spike), then preview the glued global timeline.

**Run:** GPU runtime, list the session's segments in order, *Runtime > Run all*. Download the `features_*.json` files into your BaseCampAutomation folder; the local CLI glues them and finds the cut (which may fall in any segment).

In [ ]:
# GPU runtime: Runtime > Change runtime type > T4 GPU
!pip -q install ultralytics av tqdm

In [ ]:
"""
detector.py -- pluggable person/pose detection.

Swapping the backend is a one-class change; feature code only ever sees the
`Person` dataclass. Two backends ship:

  - YoloPoseDetector : YOLO11-pose via ultralytics (AGPL-3.0; default, easiest
                       to stand up).
  - RtmPoseDetector  : RTMPose via rtmlib (Apache-2.0; prefer if this ever ships
                       in a product). RTMO (one-stage) keeps cost flat as the
                       huddle grows -- good for the dense demo crowd.

Use `build_detector(cfg)` to construct the configured backend.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional

import numpy as np


@dataclass
class Person:
    bbox: np.ndarray            # [x1, y1, x2, y2]
    conf: float
    kps: Optional[np.ndarray]   # [17, 3] COCO (x, y, score) or None


class Detector:
    """Implement detect(frame_bgr) -> List[Person]. Keep features model-agnostic."""

    def detect(self, frame_bgr: np.ndarray) -> List[Person]:
        raise NotImplementedError


class YoloPoseDetector(Detector):
    """Default detector. AGPL-3.0 -- fine for internal research; for commercial
    use prefer RTMPose/RTMO via rtmlib (Apache-2.0)."""

    def __init__(self, weights: str = "yolo11m-pose.pt", conf: float = 0.25,
                 device: Optional[str] = None):
        from ultralytics import YOLO
        self.model = YOLO(weights)
        self.conf = conf
        self.device = device

    def detect(self, frame_bgr: np.ndarray) -> List[Person]:
        r = self.model(frame_bgr, verbose=False, conf=self.conf,
                       device=self.device)[0]
        if r.boxes is None:
            return []
        xyxy = r.boxes.xyxy.cpu().numpy()
        cf = r.boxes.conf.cpu().numpy()
        kps = (r.keypoints.data.cpu().numpy()
               if r.keypoints is not None else [None] * len(xyxy))
        return [Person(bbox=b, conf=float(c), kps=k)
                for b, c, k in zip(xyxy, cf, kps)]


class RtmPoseDetector(Detector):
    """Apache-2.0 alternative via rtmlib. Best-effort: rtmlib's `Body` wrapper
    returns keypoints + scores, so the bbox is synthesized from the keypoint
    extent (slightly tighter than a true detection box -- only the posture
    aspect proxy is affected; centroid-based dispersion/density are unchanged).
    """

    def __init__(self, conf: float = 0.25, device: Optional[str] = None,
                 mode: str = "balanced", backend: str = "onnxruntime"):
        try:
            from rtmlib import Body
        except ImportError as e:  # pragma: no cover - optional dependency
            raise ImportError(
                "RTMPose backend needs: pip install rtmlib onnxruntime"
            ) from e
        self.model = Body(mode=mode, to_openpose=False, backend=backend,
                          device=device or "cpu")
        self.conf = conf

    def detect(self, frame_bgr: np.ndarray) -> List[Person]:
        keypoints, scores = self.model(frame_bgr)   # [N,17,2], [N,17]
        out: List[Person] = []
        for kp, sc in zip(keypoints, scores):
            person_conf = float(np.mean(sc))
            if person_conf < self.conf:
                continue
            x1, y1 = float(kp[:, 0].min()), float(kp[:, 1].min())
            x2, y2 = float(kp[:, 0].max()), float(kp[:, 1].max())
            kps = np.concatenate([kp, sc[:, None]], axis=1)   # [17,3]
            out.append(Person(bbox=np.array([x1, y1, x2, y2], float),
                              conf=person_conf, kps=kps))
        return out


def build_detector(cfg: dict) -> Detector:
    """Construct the detector named in cfg['backend'] ('yolo' | 'rtmpose')."""
    backend = (cfg.get("backend") or "yolo").lower()
    if backend == "yolo":
        return YoloPoseDetector(
            weights=cfg.get("weights", "yolo11m-pose.pt"),
            conf=cfg.get("conf", 0.25),
            device=cfg.get("device"),
        )
    if backend in ("rtmpose", "rtmo", "rtm"):
        return RtmPoseDetector(conf=cfg.get("conf", 0.25), device=cfg.get("device"))
    raise ValueError(f"unknown detector backend: {backend!r} (use 'yolo' or 'rtmpose')")


In [ ]:
"""
features.py -- per-frame, ROI-free, self-normalized scene-state features.

Ported from the handoff seed (starter_features.py). The math is unchanged -- it
is the proven separator between the two states -- but the tunables are now
parameters, so the pipeline can drive them from config.yaml.

  DEMO       : tight standing huddle around a stretcher
               -> LOW dispersion, HIGH crowding / local density
  DISCUSSION : participants seated in a dispersed ring
               -> HIGH dispersion, LOW crowding / local density

Everything is normalized by frame diagonal / area and by person count, so values
are comparable across resolutions, rooms, and camera placements.
"""
from __future__ import annotations

from typing import List

import cv2
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform


# --------------------------------------------------------------------------- #
# Defaults (overridable per-call; the pipeline passes values from config.yaml)
# --------------------------------------------------------------------------- #
EPS_PACK = 0.06     # NN dist (fraction of frame diagonal) below which two
                    # people count as "tightly packed" (huddle cue)
TALL_ASPECT = 1.6   # bbox height/width above which a person box is "tall"
                    # (standing people project as taller boxes in an oblique view)
KP_VIS = 0.30       # keypoint score below which a joint is not trusted

# COCO-17 keypoint indices
L_SH, R_SH, L_HIP, R_HIP, L_KNEE, R_KNEE = 5, 6, 11, 12, 13, 14


def filter_people(persons: List[Person], H: int, W: int) -> List[Person]:
    """Drop the infant manikin (and other non-people) before featurizing.

    Conservative: only removes very small, low-in-frame, horizontal boxes, so
    real (possibly seated/occluded) people are kept.
    """
    out: List[Person] = []
    for p in persons:
        x1, y1, x2, y2 = p.bbox
        bw, bh = max(x2 - x1, 1), max(y2 - y1, 1)
        area_frac = (bw * bh) / (H * W)
        horizontal = (bh / bw) < 0.6
        tiny = area_frac < 0.002
        # a tiny, horizontal box low in the frame is likely the manikin
        if tiny and horizontal and (y1 / H) > 0.45:
            continue
        out.append(p)
    return out


def extract_features(persons: List[Person], H: int, W: int, *,
                     eps_pack: float = EPS_PACK,
                     tall_aspect: float = TALL_ASPECT,
                     kp_vis: float = KP_VIS) -> dict:
    """Compute the ROI-free feature dict for one frame. See module docstring."""
    diag = (H * H + W * W) ** 0.5
    area = H * W
    n = len(persons)
    f = {"person_count": n}
    if n == 0:
        return f

    boxes = np.array([p.bbox for p in persons], dtype=float)
    cx = (boxes[:, 0] + boxes[:, 2]) / 2
    cy = (boxes[:, 1] + boxes[:, 3]) / 2
    cents = np.stack([cx, cy], axis=1)
    bw = np.clip(boxes[:, 2] - boxes[:, 0], 1, None)
    bh = boxes[:, 3] - boxes[:, 1]
    aspect = bh / bw

    # --- DISPERSION: low in the demo huddle, high in the discussion ring ------
    if n >= 3:
        try:
            f["hull_area_frac"] = float(ConvexHull(cents).volume / area)
        except Exception:
            f["hull_area_frac"] = None
    if n >= 2:
        f["mean_pairwise_norm"] = float(pdist(cents).mean() / diag)
        sq = squareform(pdist(cents))
        np.fill_diagonal(sq, np.inf)
        nn = sq.min(axis=1) / diag
        f["nn_median_norm"] = float(np.median(nn))
        f["nn_min_norm"] = float(nn.min())
        f["frac_tightly_packed"] = float(np.mean(nn < eps_pack))   # HIGH in demo

    # --- LOCAL DENSITY peak: high when a tight cluster exists (demo) ----------
    gh, gw = max(H // 20, 1), max(W // 20, 1)
    grid = np.zeros((gh, gw), np.float32)
    for x, y in cents:
        grid[min(int(y // 20), gh - 1), min(int(x // 20), gw - 1)] += 1
    grid = cv2.GaussianBlur(grid, (0, 0), 1.5)
    f["density_peak"] = float(grid.max())
    f["density_peak_per_person"] = float(grid.max() / n)

    # --- POSTURE proxies (supplementary; viewpoint-sensitive) ----------------
    f["aspect_median"] = float(np.median(aspect))
    f["frac_tall_boxes"] = float(np.mean(aspect > tall_aspect))    # standing taller

    stand = sit = usable = 0
    for p in persons:
        if p.kps is None:
            continue
        sh, hip, kn = p.kps[[L_SH, R_SH]], p.kps[[L_HIP, R_HIP]], p.kps[[L_KNEE, R_KNEE]]
        if sh[:, 2].min() < kp_vis or hip[:, 2].min() < kp_vis or kn[:, 2].min() < kp_vis:
            continue
        usable += 1
        torso = abs(hip[:, 1].mean() - sh[:, 1].mean())
        thigh = abs(kn[:, 1].mean() - hip[:, 1].mean())
        # standing: knees sit well below hips relative to torso length
        if thigh > 0.55 * max(torso, 1):
            stand += 1
        else:
            sit += 1
    f["kp_usable"], f["kp_stand"], f["kp_sit"] = usable, stand, sit

    # --- POSTURE composite: robust fraction-seated estimate (PRIMARY signal) ---
    # The Phase-1 gate showed this is the clean demo/discussion separator on real
    # footage (and it is room-geometry invariant, unlike spatial dispersion).
    # Blend keypoint sit/stand with the keypoint-free box-aspect proxy, trusting
    # keypoints in proportion to how many people have usable ones (low in the
    # occluded huddle, so the box proxy carries those frames). See FINDINGS_phase1.md.
    n_post = stand + sit
    sit_box = 1.0 - f["frac_tall_boxes"]              # taller boxes => standing
    if n_post == 0:
        f["sit_fraction"] = float(sit_box)
    else:
        sit_kp = sit / n_post
        w_kp = min(n_post / n, 1.0)                   # confidence in keypoint evidence
        f["sit_fraction"] = float(w_kp * sit_kp + (1.0 - w_kp) * sit_box)
    return f

In [ ]:
"""
frames.py -- seek-based frame sampling (handoff Stage A).

Decodes ONLY the sampled timestamps via PyAV seeking, so a 55 GB file samples
almost as fast as a small one (it reads a few percent of the bytes rather than
decoding the whole stream). Yields (timestamp_seconds, frame_bgr).
"""
from __future__ import annotations

from typing import Iterator, Optional, Tuple

import numpy as np


class FrameSource:
    """Iterable of (timestamp_seconds, frame_bgr) sampled at `fps` via seeking.

    Open once, seek many times. `start_s`/`end_s` restrict the range (used by the
    coarse->fine refinement pass). `downscale_long_edge` shrinks frames before
    they leave the source (0 = full resolution).
    """

    def __init__(self, video_path: str, fps: float = 1.0,
                 start_s: float = 0.0, end_s: Optional[float] = None,
                 downscale_long_edge: int = 0):
        self.video_path = video_path
        self.fps = float(fps)
        self.start_s = float(start_s)
        self.end_s = end_s
        self.downscale_long_edge = int(downscale_long_edge or 0)

        import av  # lazy: keep the dependency optional until sampling is used
        self._av = av
        with av.open(video_path) as c:
            vs = c.streams.video[0]
            if vs.duration is not None and vs.time_base is not None:
                self.duration_s: Optional[float] = float(vs.duration * vs.time_base)
            elif c.duration is not None:
                self.duration_s = float(c.duration / av.time_base)
            else:
                self.duration_s = None

    def _maybe_downscale(self, img: np.ndarray) -> np.ndarray:
        le = self.downscale_long_edge
        if le > 0:
            import cv2
            h, w = img.shape[:2]
            scale = le / max(h, w)
            if scale < 1.0:
                img = cv2.resize(img, (int(round(w * scale)), int(round(h * scale))),
                                 interpolation=cv2.INTER_AREA)
        return img

    def __iter__(self) -> Iterator[Tuple[float, np.ndarray]]:
        av = self._av
        end = self.end_s if self.end_s is not None else (self.duration_s or 1e9)

        targets = []
        t, step = self.start_s, 1.0 / self.fps
        while t < end:
            targets.append(t)
            t += step

        with av.open(self.video_path) as container:
            stream = container.streams.video[0]
            tb = stream.time_base
            for tt in targets:
                try:
                    container.seek(int(tt / tb), stream=stream, backward=True, any_frame=False)
                except Exception:
                    continue
                got = None
                for frame in container.decode(stream):
                    ftime = (frame.time if frame.time is not None
                             else (float(frame.pts * tb) if frame.pts is not None else tt))
                    if ftime + 1e-6 >= tt:
                        got = (ftime, frame)
                        break
                if got is None:
                    continue
                ftime, frame = got
                yield ftime, self._maybe_downscale(frame.to_ndarray(format="bgr24"))


In [ ]:
"""
motion.py -- cheap frame-difference motion signal (handoff Stage D).

Mean absolute difference between consecutive sampled frames, computed on a small
grayscale image. Detector-independent and free. The discussion is the global
motion-minimum tail, so this is a strong cross-check on the posture signal.
"""
from __future__ import annotations

import cv2
import numpy as np


def to_small_gray(frame_bgr: np.ndarray, long_edge: int = 320) -> np.ndarray:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    scale = long_edge / max(h, w)
    if scale < 1.0:
        gray = cv2.resize(gray, (int(round(w * scale)), int(round(h * scale))),
                          interpolation=cv2.INTER_AREA)
    return gray


def frame_diff(prev_gray: np.ndarray, gray: np.ndarray) -> float:
    """Mean |Δpixel| between two small gray frames; NaN if no/!=-shape previous."""
    if prev_gray is None or prev_gray.shape != gray.shape:
        return float("nan")
    return float(np.mean(np.abs(gray.astype(np.int16) - prev_gray.astype(np.int16))))


In [ ]:
"""
extract.py -- Stages A-D driver: per-frame features + motion over a whole video,
with caching.

This is the GPU-friendly half of the pipeline (detection is the expensive step).
Run it once where the model lives (Colab / a GPU box), then iterate the cheap
analysis half (smoothing/fusion/boundary) on the cached table -- tuning the
boundary logic never re-runs the model.

Cache key = (file path, size, mtime, fps, detector, conf), so re-runs skip decode
and detection automatically.
"""
from __future__ import annotations

import hashlib
import json
import os
from typing import Optional



def file_signature(path: str, fps: float, extra: str = "") -> str:
    st = os.stat(path)
    h = hashlib.sha1()
    h.update(os.path.abspath(path).encode())
    h.update(str(st.st_size).encode())
    h.update(str(int(st.st_mtime)).encode())
    h.update(f"{fps}:{extra}".encode())
    return h.hexdigest()[:16]


def extract_video(video_path: str, cfg: dict, cache_dir: str = "cache",
                  use_cache: bool = True, progress: bool = True,
                  start_s: float = 0.0, end_s: Optional[float] = None) -> dict:
    """Return {'meta': {...}, 'rows': [ {t, motion, ...features...}, ... ]}.

    Each row is one sampled frame. Spatial, posture, and motion signals are ALL
    computed here so the analysis stage can ablate/reweight them freely.
    """
    samp, det_cfg, fp = cfg["sampling"], cfg["detector"], cfg["features"]
    fps = samp["coarse_fps"]
    windowed = (start_s > 0.0) or (end_s is not None)

    sig = file_signature(video_path, fps,
                         extra=f'{det_cfg["backend"]}:{det_cfg["conf"]}'
                               f':{start_s}:{end_s}')
    cache_path = os.path.join(cache_dir, f"{os.path.basename(video_path)}.{sig}.json")
    if use_cache and os.path.exists(cache_path):
        with open(cache_path) as fh:
            return json.load(fh)

    det = build_detector(det_cfg)
    fs = FrameSource(video_path, fps=fps, start_s=start_s, end_s=end_s,
                     downscale_long_edge=samp.get("downscale_long_edge", 0))
    motion_le = cfg["motion"].get("downscale_long_edge", 320)
    drop = det_cfg.get("drop_manikin", True)

    it = fs
    if progress:
        try:
            from tqdm import tqdm
            span = (end_s or fs.duration_s or 0) - start_s
            it = tqdm(fs, total=int(span * fps) or None, desc="extract")
        except Exception:
            pass

    rows, prev_small = [], None
    for t, frame in it:
        H, W = frame.shape[:2]
        people = det.detect(frame)
        if drop:
            people = filter_people(people, H, W)
        feats = extract_features(people, H, W, eps_pack=fp["eps_pack"],
                                 tall_aspect=fp["tall_aspect"], kp_vis=fp["kp_vis"])
        small = to_small_gray(frame, motion_le)
        feats["t"] = float(t)
        feats["motion"] = frame_diff(prev_small, small)
        prev_small = small
        rows.append(feats)

    out = {
        "meta": {
            "video": os.path.basename(video_path), "fps": fps,
            "detector": det_cfg["backend"], "conf": det_cfg["conf"],
            "duration_s": fs.duration_s, "n_frames": len(rows),
            "windowed": windowed, "start_s": start_s, "end_s": end_s,
        },
        "rows": rows,
    }
    if not windowed:                      # only cache full-video passes
        os.makedirs(cache_dir, exist_ok=True)
        with open(cache_path, "w") as fh:
            json.dump(out, fh)
    return out

In [ ]:
"""
session.py -- glue per-segment feature tables into one global timeline.

The recording camera cuts out every ~30 min, so one session is several segment
files, and the demo->discussion cut can fall anywhere (including mid-segment, or
in any segment). We extract features PER SEGMENT -- so motion is never
differenced across a segment boundary (that would be a false walk-over spike) --
then concatenate the tables on a global clock and null the motion at each seam.

The raw video is only ever concatenated for the final output clips (see
cut.run_session_split); analysis works entirely on the glued feature table, so a
55 GB session never gets built into one file.
"""
from __future__ import annotations

import json


def _load(obj):
    return json.load(open(obj)) if isinstance(obj, str) else obj


def order_segments(objs, order=None):
    """Order segment feature-objects. Default: by video basename (the
    LRV_<date>_<time>_... names sort chronologically). Pass `order` (a list of
    basenames) to override."""
    if order:
        rank = {name: i for i, name in enumerate(order)}
        return sorted(objs, key=lambda o: rank.get(o["meta"].get("video", ""), 1e9))
    return sorted(objs, key=lambda o: o["meta"].get("video", ""))


def glue_features(feature_objs, order=None):
    """Concatenate per-segment feature tables into one global-timeline table.

    Returns {meta, rows (global t, with a 'seg' index), segments [{idx, video,
    duration_s, offset_s}]}.
    """
    objs = order_segments([_load(o) for o in feature_objs], order)
    glued, segments, offset = [], [], 0.0
    for idx, o in enumerate(objs):
        meta, rows = o["meta"], o["rows"]
        dur = float(meta.get("duration_s") or (rows[-1]["t"] + 1.0 if rows else 0.0))
        for j, r in enumerate(rows):
            gr = dict(r)
            gr["t"] = float(r["t"]) + offset
            gr["seg"] = idx
            if j == 0:
                gr["motion"] = float("nan")        # no frame-diff across the seam
            glued.append(gr)
        segments.append({"idx": idx, "video": meta.get("video"),
                         "duration_s": dur, "offset_s": offset})
        offset += dur
    return {"meta": {"video": "SESSION", "fps": objs[0]["meta"].get("fps", 1.0) if objs else 1.0,
                     "duration_s": offset, "n_segments": len(segments)},
            "rows": glued, "segments": segments}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# List this session's segments IN ORDER (the ~30-min files from one recording).
# For a single-file session, just list the one file.
VIDEO_PATHS = [
    "/content/drive/MyDrive/sim_samples/SEGMENT_1.mp4",   # <-- EDIT
    # "/content/drive/MyDrive/sim_samples/SEGMENT_2.mp4",
    # "/content/drive/MyDrive/sim_samples/SEGMENT_3.mp4",
]
import os
for p in VIDEO_PATHS:
    assert os.path.exists(p), f"not found: {p}"
print(len(VIDEO_PATHS), "segment(s)")


In [ ]:
import json, os
CFG = {
    "sampling": {"coarse_fps": 1.0, "downscale_long_edge": 1280},
    "detector": {"backend": "yolo", "weights": "yolo11m-pose.pt",
                 "conf": 0.25, "device": None, "drop_manikin": True},
    "features": {"eps_pack": 0.06, "tall_aspect": 1.6, "kp_vis": 0.30},
    "motion":   {"downscale_long_edge": 320},
}
objs, out_files = [], []
for vp in VIDEO_PATHS:
    print("extracting:", os.path.basename(vp))
    out = extract_video(vp, CFG, cache_dir="cache", use_cache=True)
    objs.append(out)
    fn = f"features_{os.path.splitext(os.path.basename(vp))[0]}.json"
    json.dump(out, open(fn, "w")); out_files.append(fn)
    print(f"  {out['meta']['n_frames']} frames, {(out['meta']['duration_s'] or 0)/60:.1f} min -> {fn}")


In [ ]:
import numpy as np, matplotlib.pyplot as plt
g = glue_features(objs)
rows, segs = g["rows"], g["segments"]
t = np.array([r["t"] for r in rows]) / 60.0
def series(k): return np.array([r.get(k, np.nan) for r in rows], float)
def smooth(x, k=15):
    n, h, o = len(x), k // 2, np.full(len(x), np.nan)
    for i in range(n):
        seg = x[max(0, i-h):min(n, i+h+1)]; seg = seg[~np.isnan(seg)]
        if len(seg): o[i] = seg.mean()
    return o
sit, mot = series("sit_fraction"), series("motion")
mn = mot / np.nanmax(mot) if np.nanmax(mot) > 0 else mot
fig, ax = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
ax[0].plot(t, smooth(sit), lw=2); ax[0].set_ylim(0, 1); ax[0].set_ylabel("sit_fraction")
ax[0].set_title("SESSION posture (glued); dotted = segment seams")
ax[1].plot(t, smooth(mn), lw=2); ax[1].set_ylabel("motion (norm)"); ax[1].set_xlabel("minutes")
ax[1].set_title("SESSION motion -- expect ONE walk-over spike at the demo->discussion cut")
for s in segs[1:]:
    for a in ax: a.axvline(s["offset_s"] / 60, c="0.6", ls=":", lw=.9)
plt.tight_layout(); plt.show()
print(f"session: {g['meta']['duration_s']/60:.1f} min over {len(segs)} segment(s)")


In [ ]:
# Drop these features_*.json into your BaseCampAutomation folder; the local CLI
# globs them (it re-glues + finds the cut, which may span segments).
from google.colab import files
for fn in out_files:
    files.download(fn)
